# Multi-Layer Perceptron (MLP) — Iris Dataset

This notebook follows the same simple approach as the single-layer perceptron notebook.

**Goal:** Use an MLP to identify whether an Iris flower is **Setosa** or **Not Setosa**.

**Dataset file:** `iris.csv` (upload the same Kaggle Iris CSV used in the perceptron notebook).

- Input features: `PetalLengthCm`, `PetalWidthCm`
- Target: `Species`
- Binary target: Setosa = 1, other species = 0
- MLP: two hidden layers

In [1]:
from google.colab import files

uploaded = files.upload()

ModuleNotFoundError: No module named 'google.colab'

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("iris.csv")
df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             150 non-null    int64  
 1   SepalLengthCm  150 non-null    float64
 2   SepalWidthCm   150 non-null    float64
 3   PetalLengthCm  150 non-null    float64
 4   PetalWidthCm   150 non-null    float64
 5   Species        150 non-null    object 
dtypes: float64(4), int64(1), object(1)
memory usage: 7.2+ KB


In [4]:
df.isnull().sum()

Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64

In [5]:
df = df.drop(["Id"], axis=1, errors="ignore")
df.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [6]:
df["Species"].value_counts()

Species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

In [7]:
# Convert the target labels to shorter names (if needed)
df["Species"] = df["Species"].replace({
    "Iris-setosa": "Setosa",
    "Iris-virginica": "Virginica",
    "Iris-versicolor": "Versicolor"
})

df["Species"].value_counts()

Species
Setosa        50
Versicolor    50
Virginica     50
Name: count, dtype: int64

In [8]:
# Select two input features
X = df[["PetalLengthCm", "PetalWidthCm"]]
X.head()

,PetalLengthCm,PetalWidthCm
0,1.4,0.2
1,1.4,0.2
2,1.3,0.2
3,1.5,0.2
4,1.4,0.2


In [9]:
# Select the target column
y = df["Species"]

# Convert multiclass target to binary:
# Setosa = 1, Versicolor and Virginica = 0
y = (y == "Setosa").astype(int)

y.value_counts()

Species
0    100
1     50
Name: count, dtype: int64

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((120, 2), (30, 2))

## Build and train the Multi-Layer Perceptron

`hidden_layer_sizes=(5, 3)` means the network has:
- Hidden layer 1: 5 neurons
- Hidden layer 2: 3 neurons

The MLP learns its weights using backpropagation and the Adam optimizer. We scale the features because neural networks generally work better when numeric inputs are on similar scales.

In [11]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

model = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(5, 3),
        activation="relu",
        solver="adam",
        max_iter=1000,
        random_state=42
    )
)

model.fit(X_train, y_train)

C:\Users\hasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Pipeline(steps=[('standardscaler', StandardScaler()),
                ('mlpclassifier',
                 MLPClassifier(hidden_layer_sizes=(5, 3), max_iter=1000,
                               random_state=42))])

In [12]:
# Predict the test data
y_pred = model.predict(X_test)

y_pred

array([0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1,
       0, 1, 1, 0, 0, 0, 1, 0])

In [13]:
# Compare predictions with actual values
print("Predicted:", y_pred)
print("Actual:   ", y_test.to_numpy())

Predicted: [0 1 0 0 0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 1 0 1 0 1 1 0 0 0 1 0]
Actual:    [0 1 0 0 0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 1 0 1 0 1 1 0 0 0 1 0]


In [14]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc * 100:.2f}%")

print("\\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Not Setosa", "Setosa"]))

Accuracy: 100.00%
\nConfusion matrix:
[[20  0]
 [ 0 10]]
\nClassification report:
              precision    recall  f1-score   support

  Not Setosa       1.00      1.00      1.00        20
      Setosa       1.00      1.00      1.00        10

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



## What changed from the single-layer perceptron?

- Single-layer perceptron: `Perceptron()`
- Multi-layer perceptron: `MLPClassifier(hidden_layer_sizes=(5, 3))`
- The MLP has **two hidden layers** with 5 and 3 neurons.
- The rest of the workflow remains familiar: prepare `X` and `y`, split the data, fit the model, predict, and evaluate.

**Note:** This is a teaching example, not a production model. The notebook predicts Setosa versus the other two species, rather than distinguishing all three species separately.